## Equation (26) from the paper "Double-bracket quantum algorithms for high-fidelity ground state preparation"

$$\hat{H}_k(s) = \exp{(s[\hat{D}, \hat{H}_k])}\cdot \hat{H}_k(0)\cdot\exp{(-s[\hat{D}, \hat{H}_k])}$$

with
* $k$ DBQA step 
* $s$ time evolution

#### Lie Bracket Recursion and Rotation Unitaries from 0 to k

$$\hat{H}_k = \exp{(s_k[\hat{D}, \hat{H}_{k-1}])}\cdot \hat{H}_{k-1}\cdot\exp{(-s_k[\hat{D}, \hat{H}_{k-1}])}$$
$$\hat{U}_k = \hat{U}_{k-1}\cdot\exp(-s_k[\hat{D},\hat{H}_{k-1}])$$
$$\hat{H}_k = \hat{U}_k^{\dagger}\hat{H}_0\hat{U}_k$$

* $s_k$ step duration

---

#### Approximation

$$\exp(-s[\hat{D}, \hat{H}_{k-1}]) \approx \exp(i\sqrt{s}\hat{D})\cdot \hat{U}_{k-1}^{\dagger}\cdot\exp(i\sqrt{s}\hat{H}_0) \cdot\hat{U}_{k-1}\cdot\exp(-i\sqrt{s}\hat{D})$$

---

#### Warm Start
For the first step we have
$$\hat{H}_{\text{ws}}=\hat{U}_0^{\dagger}\hat{H}_0\hat{U}_0$$
It holds 
$$\hat{H}_k(s_k)=\hat{H}_{k+1}\Rightarrow  \hat{H}_k(0)=\hat{H}_{k-1}(s_k)=\hat{H}_k$$
this means
$$\hat{H}_0(s_k)=\hat{H}_{\text{ws}}$$
$$\Rightarrow \hat{U}_0 = \exp(-s_k[\hat{D},\hat{H}_0])$$
For the start we set 
$$\hat{H}_0 = \hat{H}_{XXZ}$$

---

## Choices for $H$ and $D$

#### $\hat{D}$: Classical Ising Model

Parametrize $\hat{D}$ as classical Ising model with nearest neighbor (NN) interactions, and optimize in each step $k$, i.e.

$$\hat{D}_k(B^{(k)}, J^{(k)})=\sum_{i=1}^{L} ( B _i^{(k)}\hat{Z}_i+J^{(k)}_i\hat{Z}_{i+1}\hat{Z}_i )$$

with
* $L=$ number of sites in the chain and 
* $\{B _i\}, \{J _i\}$, $i=1,...,L$ parametrization coefficients, optimized in each step $k$

For the beginning, choose a toy model for $\hat{D}$ some integer diagonal matrix.

---

#### Choice of Hamiltonian: XXZ Heisenberg Model on 1D lattice with $L$ qubits

Replace the spin by quantum operators $A\in \{X,Y,Z\}$ of dimension $2^L\times 2^L$ acting upon the tensor product space $(\mathbb {C} ^{2})^{\otimes L}$. This naturally fits to a 1D lattice of $L$ qubits.
* $I$ is the $2\times 2$ identity matrix
* $\sigma^{A}$ the $2\times 2$ Pauli matrices

$$\hat{A} _{i}=I^{\otimes i-1}\otimes \sigma ^{A}\otimes I^{\otimes L-i}$$

$$\hat{A} _{1}=\sigma ^{A}\otimes I \otimes ... \otimes I$$
$$\hat{A} _{L}=I \otimes ... \otimes I \otimes \sigma ^{A}$$



With these identity matrices in place, $\sigma^{A}$ attacks exactly the qubit at 1D lattice position $i$ and leaves all other quibts invariant.

The Hamiltonian is then 
$$\hat{H}_{XXZ}=\sum_{i=1}^{L} (\hat{X}_{i}\hat{X}_{i+1}+\hat{Y}_{i}\hat{Y}_{i+1}+\Delta \hat{Z}_{i}\hat{Z}_{i+1})$$

with 
* index $i$: Pauli Operator acting on $i$-th qubit
* $\Delta$: anisotropy parameter (measure for gap(less) system)

Boundary Conditions: 
* periodic: $X_{L+1}=X_1$ &rarr; VQE
* open: $X_{L+1}=0$ &rarr; more natural on hardware and larger systems

---

## `Code` Set-Up -- XXZ Hamiltonian and toy model D 

Install `numpy` and `matplotlib` and `scipy`.

Choose for XXX open boundary conditions $X_{L+1}=0$
* $L=20$ qubits
* $\Delta = 1$

Choose for XXZ periodic boundary conditions $X_{L+1}=X_1$
* $L=10$ qubits
* $\Delta = 0.5$

In [1]:
# Python Packages
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

In [2]:
# Parameters and function

L = 2
Delta = 0.5


sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

def commutator(A, B):
    return A @ B - B @ A

In [3]:
# Pauli Operators and Hamiltonian XXZ for periodic boundary conditions

def X( pos_i ):
    if pos_i == 1:
        return np.kron( sigma_x, np.eye(int(2**(L- 1))) )
    if pos_i == L:
        return np.kron( np.eye(int(2**(L- 1))), sigma_x )
    if pos_i == L+1:
        return X(1)
    return np.kron( np.kron(np.eye(int(2**(pos_i -1))), sigma_x), np.eye(int(2**(L- pos_i))) ) # Spin operator at position pos_i
def Y( pos_i ):
    if pos_i == 1:
        return np.kron( sigma_y, np.eye(int(2**(L- 1))) )
    if pos_i == L:
        return np.kron( np.eye(int(2**(L- 1))), sigma_y )
    if pos_i == L+1:
        return Y(1)
    return np.kron( np.kron(np.eye(int(2**(pos_i -1))), sigma_y), np.eye(int(2**(L- pos_i))) ) # Spin operator at position pos_i
def Z( pos_i ):
    if pos_i == 1:
        return np.kron( sigma_z, np.eye(int(2**(L- 1))) )
    if pos_i == L:
        return np.kron( np.eye(int(2**(L- 1))), sigma_z )
    if pos_i == L+1:
        return Z(1)
    return np.kron( np.kron(np.eye(int(2**(pos_i -1))), sigma_z), np.eye(int(2**(L- pos_i))) ) # Spin operator at position pos_i

H_XXZ = sum([ X(i) @ X(i+1) + Y(i) @ Y(i+1) + Delta * Z(i) @ Z(i+1)  for i in range(1, L+1)])

print ("Hamiltonian H_XXZ = \n", H_XXZ)

Hamiltonian H_XXZ = 
 [[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j  4.+0.j  0.+0.j]
 [ 0.+0.j  4.+0.j -1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]


In [4]:
# Toy Model D

diagonal_values = range(1, 2**L + 1)  
D_matrix = np.diag(diagonal_values)

print("Diagonal matrix D = \n", D_matrix)

Diagonal matrix D = 
 [[1 0 0 0]
 [0 2 0 0]
 [0 0 3 0]
 [0 0 0 4]]


## `Code` for equation 26



In [5]:
# Implementing warm start and exponentials

s_step = 0.2 # not sure what to choose for this, but it should be small enough to ensure convergence

def exp_D(s):
    return expm(1j * np.sqrt(s) * D_matrix)
def exp_D_inv(s):
    return expm(-1j * np.sqrt(s) * D_matrix)
def exp_H_0(s):
    return expm(1j * np.sqrt(s) * H_XXZ)

U_0 = expm(-s_step * commutator(D_matrix, H_XXZ))
H_ws = U_0.conj().T @ H_XXZ @ U_0

print("Warm-starting unitary U_0 = \n", U_0)
print("Warm-started Hamiltonian H_ws = \n", H_ws)

Warm-starting unitary U_0 = 
 [[ 1.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  0.69670671+0.j  0.71735609+0.j  0.        +0.j]
 [ 0.        +0.j -0.71735609+0.j  0.69670671+0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j  1.        +0.j]]
Warm-started Hamiltonian H_ws = 
 [[ 1.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j -4.99829441+0.j -0.11679809+0.j  0.        +0.j]
 [ 0.        +0.j -0.11679809+0.j  2.99829441+0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j  1.        +0.j]]


In [6]:
# Implementing Unitary and exponential approximation

def U(k):
    if k == 0:
        return U_0 # Warm start unitary
    else:
        for i in range(0, k+1): # for loop to implement recursion
            if i == 0:
                U_current = U_0
                #print(f"Loop {i}: Unitary U({i})")
            else:
                U_current = U_current @ exp_D(s_step) @ U_current.conj().T @ exp_H_0(s_step) @ U_current @ exp_D_inv(s_step)
                # print(f"Loop {i}: Unitary U({i})")
        return U_current

def exp_approx(k, s):
    return exp_D(s) @ U(k).conj().T @ exp_H_0(s) @ U(k) @ exp_D_inv(s)

print(U_0 == U(0))
print("After Calling Unitary U(1) = \n", U(1))
print("Exponential approximation exp_approx(1, 0.2) = \n", exp_approx(1, 0.2))

[[ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]]
After Calling Unitary U(1) = 
 [[ 0.9016556 +0.43245484j  0.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j         -0.42993273-0.56832231j  0.14733283+0.68590133j
   0.        +0.j        ]
 [ 0.        +0.j          0.44267531+0.54424826j  0.17408342+0.69103355j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j
   0.9016556 +0.43245484j]]
Exponential approximation exp_approx(1, 0.2) = 
 [[ 0.9016556 +0.43245484j  0.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j         -0.61695962-0.786096j   -0.03757209+0.00149508j
   0.        +0.j        ]
 [ 0.        +0.j          0.02468477-0.02836475j  0.22684197+0.97320544j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j
   0.9016556 +0.43245484j]]


In [ ]:
# Implementing equation 26 as a function of k and s, with warm start

def H(k, s):
    if k == 0:
        return H_XXZ # Initial Hamiltonian
    if k == 1:
        return H_ws # Warm start Hamiltonian
    else:
        for i in range(0, k+1): # for loop to implement recursion
            if i == 0:
                H_step = H_XXZ
                print(f"Loop zero: Hamiltonian H({i})")
            if i == 1:
                H_step = H_ws
                print(f"Loop one: Hamiltonian H({i})")
            if i > 1:
                rec = i - 1
                H_step = exp_approx(rec, s_step).conj().T @ H_step @ exp_approx(rec, s_step) # Call previous H at time s_step
                H_current = exp_approx(i, s).conj().T @ H_step @ exp_approx(i, s)  # I am pretty sure this is not entirely correct, because I need to call the previous H at time s_step
                print(f"Loop {i}: Hamiltonian H({i}, {s})")
        return H_current

H(4, 0.1)

Loop zero: Hamiltonian H(0)
Loop one: Hamiltonian H(1)
Loop 2: Hamiltonian H(2, 0.1)
Loop 3: Hamiltonian H(3, 0.1)
Loop 4: Hamiltonian H(4, 0.1)


array([[ 1.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  0.        +0.00000000e+00j],
       [ 0.        +0.00000000e+00j, -4.97888558+4.57966998e-16j,
         0.36379264-1.90064439e-01j,  0.        +0.00000000e+00j],
       [ 0.        +0.00000000e+00j,  0.36379264+1.90064439e-01j,
         2.97888558+2.22044605e-16j,  0.        +0.00000000e+00j],
       [ 0.        +0.00000000e+00j,  0.        +0.00000000e+00j,
         0.        +0.00000000e+00j,  1.        -5.55111512e-17j]])

## Some plot ?

In [8]:
s_grid = np.linspace(0, 0.2, 10) # no idea what to choose here
H_vals = [H(1, s) for s in s_grid] # you cannot plot this but you may plot the norm of this, or the trace, or the eigenvalues, or something else that is a scalar value
print(s_grid)
print(H_vals)

# plt.figure(figsize=(7, 4))
# plt.plot(s_grid, H_vals, label="some other title")
# plt.xlabel("s"); plt.ylabel("")
# plt.title("")
# plt.show()

[0.         0.02222222 0.04444444 0.06666667 0.08888889 0.11111111
 0.13333333 0.15555556 0.17777778 0.2       ]
[array([[ 1.        +0.j,  0.        +0.j,  0.        +0.j,
         0.        +0.j],
       [ 0.        +0.j, -4.99829441+0.j, -0.11679809+0.j,
         0.        +0.j],
       [ 0.        +0.j, -0.11679809+0.j,  2.99829441+0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.        +0.j,  0.        +0.j,
         1.        +0.j]]), array([[ 1.        +0.j,  0.        +0.j,  0.        +0.j,
         0.        +0.j],
       [ 0.        +0.j, -4.99829441+0.j, -0.11679809+0.j,
         0.        +0.j],
       [ 0.        +0.j, -0.11679809+0.j,  2.99829441+0.j,
         0.        +0.j],
       [ 0.        +0.j,  0.        +0.j,  0.        +0.j,
         1.        +0.j]]), array([[ 1.        +0.j,  0.        +0.j,  0.        +0.j,
         0.        +0.j],
       [ 0.        +0.j, -4.99829441+0.j, -0.11679809+0.j,
         0.        +0.j],
       [ 0.        +0.j, -0.1167

## Computation of Relative Energy Difference

$$1 - \frac{\tilde{E}_0}{E_0}$$

with 
* target / true ground state energy $E_0$ and 
* actually achieved energy / ground state energy approximation $\tilde{E}_0$

In [9]:
# to be implemented

## Using the actual Ising model for D (and not a toy model)

In [10]:
# to be implemented

## Finding an optimal step duration via greediness

In [11]:
# to be implemented